In [1]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import math as m
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as imbPipeline
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, PowerTransformer, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import auc,accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix,roc_auc_score,roc_curve,RocCurveDisplay
import warnings
from sklearn.model_selection import GridSearchCV,train_test_split
from sklearn.preprocessing import StandardScaler

warnings.simplefilter("always", category=FutureWarning)

In [2]:
titanicdf = sns.load_dataset("titanic")
titanicdf.shape
titanicdf.info()
titanicdf.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.7+ KB


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [3]:
titanicdf.isnull().sum()

,0
survived,0
pclass,0
sex,0
age,177
sibsp,0
parch,0
fare,0
embarked,2
class,0
who,0


In [4]:
titanicdf.drop(columns=["pclass", "embarked", "who", "adult_male", "deck", "alive", "alone"], inplace=True)
titanicdf.shape
titanicdf.info()
titanicdf.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   sex          891 non-null    object  
 2   age          714 non-null    float64 
 3   sibsp        891 non-null    int64   
 4   parch        891 non-null    int64   
 5   fare         891 non-null    float64 
 6   class        891 non-null    category
 7   embark_town  889 non-null    object  
dtypes: category(1), float64(2), int64(3), object(2)
memory usage: 49.9+ KB


,survived,sex,age,sibsp,parch,fare,class,embark_town
0,0,male,22.0,1,0,7.2500,Third,Southampton
1,1,female,38.0,1,0,71.2833,First,Cherbourg
2,1,female,26.0,0,0,7.9250,Third,Southampton
3,1,female,35.0,1,0,53.1000,First,Southampton
4,0,male,35.0,0,0,8.0500,Third,Southampton


In [5]:
titanicdf.dropna(inplace=True)
titanicdf.shape
titanicdf.info()
titanicdf.head()

<class 'pandas.core.frame.DataFrame'>
Index: 712 entries, 0 to 890
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     712 non-null    int64   
 1   sex          712 non-null    object  
 2   age          712 non-null    float64 
 3   sibsp        712 non-null    int64   
 4   parch        712 non-null    int64   
 5   fare         712 non-null    float64 
 6   class        712 non-null    category
 7   embark_town  712 non-null    object  
dtypes: category(1), float64(2), int64(3), object(2)
memory usage: 45.3+ KB


,survived,sex,age,sibsp,parch,fare,class,embark_town
0,0,male,22.0,1,0,7.2500,Third,Southampton
1,1,female,38.0,1,0,71.2833,First,Cherbourg
2,1,female,26.0,0,0,7.9250,Third,Southampton
3,1,female,35.0,1,0,53.1000,First,Southampton
4,0,male,35.0,0,0,8.0500,Third,Southampton


In [6]:
titanicdf.describe().T

,count,mean,std,min,25%,50%,75%,max
survived,712.0,0.404494,0.491139,0.00,0.00,0.00000,1.0,1.0000
age,712.0,29.642093,14.492933,0.42,20.00,28.00000,38.0,80.0000
sibsp,712.0,0.514045,0.930692,0.00,0.00,0.00000,1.0,5.0000
parch,712.0,0.432584,0.854181,0.00,0.00,0.00000,1.0,6.0000
fare,712.0,34.567251,52.938648,0.00,8.05,15.64585,33.0,512.3292


In [7]:
titanicdf_num = titanicdf.select_dtypes(include='number')

nCol = 3
nRow = m.ceil(len(titanicdf_num.columns) / nCol)
fig = make_subplots(rows=nRow, cols=nCol, subplot_titles=[colname.title() for colname in titanicdf_num.columns])

for i, col in enumerate(titanicdf_num.columns):
    fig.add_trace(go.Histogram(x=titanicdf_num[col], name=col), row=(i // nCol) + 1, col=(i % nCol) + 1)

fig.update_layout(height=500, width=900, showlegend=False,title_text="Distributions of Numerical Variables",)
fig.show()

In [8]:
titanicdf["class"] = titanicdf["class"].astype("object")
titanicdf_obj = titanicdf.select_dtypes(include='object')

nCol = 2
nRow = m.ceil(len(titanicdf_obj.columns) / nCol)
fig = make_subplots(rows=nRow, cols=nCol, subplot_titles=[colname.title() for colname in titanicdf_obj.columns])

for i, col in enumerate(titanicdf_obj.columns):
    fig.add_trace(go.Histogram(x=titanicdf_obj[col], name=col), row=(i // nCol) + 1, col=(i % nCol) + 1)

fig.update_layout(height=500, width=900, showlegend=False,title_text="Distributions of Object Variables",)
fig.show()

In [37]:
fig = px.scatter_matrix(titanicdf_num,
                        dimensions=titanicdf_num.columns.tolist(),
                        color="survived",
                        opacity=0.5)
fig.update_layout(title="Pairwise Scatter Plot Matrix",
                  width=800,
                  height=800
                  )
fig.show()

In [9]:
titanicdf_num_Long = titanicdf_num.melt(id_vars="survived",var_name="variable", value_name="value")
fig = px.box(titanicdf_num_Long,
             x="variable",
             y="value",
             color="survived")
fig.update_layout(title="Box Plot of Numerical Variables")
fig.show()

In [10]:
corrols = titanicdf_num.corr()
fig = px.imshow(corrols,text_auto=True,color_continuous_scale='RdBu_r',title="Correlation Matrix Heatmap")
fig.show()

In [11]:
for colname in titanicdf.columns:
  print(colname)
  print(titanicdf[colname].unique(),"\n")

survived
[0 1] 

sex
['male' 'female'] 

age
[22.   38.   26.   35.   54.    2.   27.   14.    4.   58.   20.   39.
 55.   31.   34.   15.   28.    8.   19.   40.   66.   42.   21.   18.
  3.    7.   49.   29.   65.   28.5   5.   11.   45.   17.   32.   16.
 25.    0.83 30.   33.   23.   24.   46.   59.   71.   37.   47.   14.5
 70.5  32.5  12.    9.   36.5  51.   55.5  40.5  44.    1.   61.   56.
 50.   36.   45.5  20.5  62.   41.   52.   63.   23.5   0.92 43.   60.
 10.   64.   13.   48.    0.75 53.   57.   80.   70.   24.5   6.    0.67
 30.5   0.42 34.5  74.  ] 

sibsp
[1 0 3 4 2 5] 

parch
[0 1 2 5 3 4 6] 

fare
[  7.25    71.2833   7.925   53.1      8.05    51.8625  21.075   11.1333
  30.0708  16.7     26.55    31.275    7.8542  16.      29.125   18.
  26.      13.       8.0292  35.5     31.3875 263.      27.7208  10.5
  82.1708  52.      11.2417   9.475   21.      41.5792   7.8792  17.8
  39.6875   7.8     76.7292  61.9792   7.2292  27.75    46.9     83.475
  27.9      8.1583   8

In [12]:
def relab(X):
  X = X.replace({"male":1,"female":0})
  return X

binarPipe = Pipeline(steps=[
    ('lab', FunctionTransformer(relab))
])
catiPipe = Pipeline(steps=[
    ('oneHot', OneHotEncoder(handle_unknown='ignore',sparse_output=False))
])
numPipe = Pipeline(steps=[
    ('yeojo', PowerTransformer(method="yeo-johnson",standardize=True))
])

y_vals = titanicdf["survived"]
X_vals = titanicdf.drop(columns=["survived"])

numCol = X_vals.select_dtypes(include='number').columns.tolist()
catCol = X_vals.select_dtypes(include='object').columns.tolist()
catCol.remove("sex")
binaCol = ["sex"]

preprocessor = ColumnTransformer(transformers=[
    ('binary', binarPipe, binaCol),
    ('categorical', catiPipe, catCol),
    ('numerical', numPipe, numCol)
])

titanicdf_prepro = preprocessor.fit_transform(X_vals)
onh_colnames = list(preprocessor.named_transformers_['categorical'].named_steps['oneHot'].get_feature_names_out(catCol))
print(binaCol)
print(onh_colnames)
print(numCol)
titanicdf_prepro = pd.DataFrame(titanicdf_prepro,columns=binaCol+onh_colnames+numCol)

titanicdf_prepro.head()

['sex']
['class_First', 'class_Second', 'class_Third', 'embark_town_Cherbourg', 'embark_town_Queenstown', 'embark_town_Southampton']
['age', 'sibsp', 'parch', 'fare']


<ipython-input-12-0ef783d30014>:2: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



,sex,class_First,class_Second,class_Third,embark_town_Cherbourg,embark_town_Queenstown,embark_town_Southampton,age,sibsp,parch,fare
0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,-0.469528,1.290366,-0.609138,-0.997650
1,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.609720,1.290366,-0.609138,1.287757
2,0.0,0.0,0.0,1.0,0.0,0.0,1.0,-0.186239,-0.716478,-0.609138,-0.901073
3,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.417070,1.290366,-0.609138,1.024276
4,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.417070,-0.716478,-0.609138,-0.884112


In [13]:
X_train, X_test, y_train, y_test = train_test_split(titanicdf_prepro, y_vals, test_size=0.20, random_state=42)

resamPipe = imbPipeline([
    ('under', RandomUnderSampler(sampling_strategy=0.66)),
    ('over', SMOTE(sampling_strategy=1))
])
X_train_resam, y_train_resam = resamPipe.fit_resample(X_train, y_train)

In [14]:
y_train_resam.value_counts()

,count
survived,
0,340
1,340


In [15]:
log_model = LogisticRegression()

param_grid = {
    'C': [0.01, 0.1, 1],
    'solver': ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'],
    'penalty': ['l1', 'l2', 'elasticnet', None],
    'max_iter': [200,300,500,1000],
}

grid_model = GridSearchCV(log_model, param_grid, cv=5, scoring='accuracy',n_jobs=-1)

log_model.fit(X_train_resam, y_train_resam)
grid_model.fit(X_train_resam, y_train_resam)

gridOpti_model = grid_model.best_estimator_
gridOpti_params = grid_model.best_params_

/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning:


660 fits failed out of a total of 1440.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
60 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.11/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py", line 1193, in fit
    solver = _check_so

In [16]:
print(f"Best Hyperparameters: {gridOpti_params}")

Best Hyperparameters: {'C': 0.1, 'max_iter': 200, 'penalty': 'l2', 'solver': 'lbfgs'}


In [38]:
y_log_pred_train = log_model.predict(X_train_resam)
y_grd_pred_train = gridOpti_model.predict(X_train_resam)

def evaluateModel(modelName,y_pred,y_true):
  accuracy = accuracy_score(y_true,y_pred)
  precision = precision_score(y_true,y_pred)
  recall = recall_score(y_true,y_pred)
  f1 = f1_score(y_true,y_pred)

  print(f"Model {modelName} is evaluated!")
  return [accuracy,precision,recall,f1]

modelPerf = {
    "model_name":[],
    "accuracy":[],
    "precision":[],
    "recall":[],
    "f1":[]
}

accuracy,precision,recall,f1 = evaluateModel("Logistic Regression",y_log_pred_train,y_train_resam)
modelPerf["model_name"].append("Logistic Regression")
modelPerf["accuracy"].append(accuracy)
modelPerf["precision"].append(precision)
modelPerf["recall"].append(recall)
modelPerf["f1"].append(f1)

accuracy,precision,recall,f1 = evaluateModel("Hypertuned Logistic Regression",y_grd_pred_train,y_train_resam)
modelPerf["model_name"].append("Hypertuned Logistic Regression")
modelPerf["accuracy"].append(accuracy)
modelPerf["precision"].append(precision)
modelPerf["recall"].append(recall)
modelPerf["f1"].append(f1)

modelPerf_df = pd.DataFrame(modelPerf)
modelPerf_df

Model Logistic Regression is evaluated!
Model Hypertuned Logistic Regression is evaluated!


,model_name,accuracy,precision,recall,f1
0,Logistic Regression,0.795588,0.791304,0.802941,0.797080
1,Hypertuned Logistic Regression,0.798529,0.792507,0.808824,0.800582


In [18]:
log_model_classRep = classification_report(y_train_resam,y_log_pred_train)
grd_model_classRep = classification_report(y_train_resam,y_grd_pred_train)

print(f"Logistic Regression Classification Report:\n{log_model_classRep}")
print(f"Hypertuned Logistic Regression Classification Report:\n{grd_model_classRep}")

Logistic Regression Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.79      0.79       340
           1       0.79      0.80      0.80       340

    accuracy                           0.80       680
   macro avg       0.80      0.80      0.80       680
weighted avg       0.80      0.80      0.80       680

Hypertuned Logistic Regression Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.79      0.80       340
           1       0.79      0.81      0.80       340

    accuracy                           0.80       680
   macro avg       0.80      0.80      0.80       680
weighted avg       0.80      0.80      0.80       680



In [19]:
confusMat_train = confusion_matrix(y_log_pred_train,y_train_resam)
fig = px.imshow(confusMat_train,text_auto=True,color_continuous_scale='RdBu_r',title="Confusion Matrix Heatmap Logistic Regression (Train)")
fig.show()

In [20]:
confusMat_train = confusion_matrix(y_grd_pred_train,y_train_resam)
fig = px.imshow(confusMat_train,text_auto=True,color_continuous_scale='RdBu_r',title="Confusion Matrix Heatmap Hypertuned Logistic Regression (Train)")
fig.show()

In [21]:
y_train_log_scores = log_model.predict_proba(X_train_resam)[:, 1]
y_train_grd_scores = gridOpti_model.predict_proba(X_train_resam)[:, 1]

fpr_log, tpr_log, thresholds_log = roc_curve(y_train_resam, y_train_log_scores)
auc_train_log_score = auc(fpr_log, tpr_log)
fpr_grd, tpr_grd, thresholds_grd = roc_curve(y_train_resam, y_train_grd_scores)
auc_train_grd_score = auc(fpr_grd, tpr_grd)

In [22]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=fpr_log, y=tpr_log, mode='lines', name='ROC Curve'))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Blind Guess', line=dict(dash='dash')))

fig.update_layout(
    title=f'Logistic Regression ROC Curve for Train Data (AUC = {round(auc_train_log_score, 3)})',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    width=600,
    height=400
)

In [23]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=fpr_grd, y=tpr_grd, mode='lines', name='ROC Curve'))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Blind Guess', line=dict(dash='dash')))

fig.update_layout(
    title=f'Hypertuned Logistic Regression ROC Curve for Train Data (AUC = {round(auc_train_grd_score, 3)})',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    width=600,
    height=400
)

In [24]:
y_log_pred_test = log_model.predict(X_test)
y_grd_pred_test = gridOpti_model.predict(X_test)

modelPerfTest = {
    "model_name":[],
    "accuracy":[],
    "precision":[],
    "recall":[],
    "f1":[]
}

accuracy,precision,recall,f1 = evaluateModel("Logistic Regression",y_log_pred_test,y_test)
modelPerfTest["model_name"].append("Logistic Regression")
modelPerfTest["accuracy"].append(accuracy)
modelPerfTest["precision"].append(precision)
modelPerfTest["recall"].append(recall)
modelPerfTest["f1"].append(f1)

accuracy,precision,recall,f1 = evaluateModel("Hypertuned Logistic Regression",y_grd_pred_test,y_test)
modelPerfTest["model_name"].append("Hypertuned Logistic Regression")
modelPerfTest["accuracy"].append(accuracy)
modelPerfTest["precision"].append(precision)
modelPerfTest["recall"].append(recall)
modelPerfTest["f1"].append(f1)

modelPerfTest_df = pd.DataFrame(modelPerfTest)
modelPerfTest_df

Model Logistic Regression is evaluated!
Model Hypertuned Logistic Regression is evaluated!


,model_name,accuracy,precision,recall,f1
0,Logistic Regression,0.811189,0.800000,0.761905,0.780488
1,Hypertuned Logistic Regression,0.804196,0.777778,0.777778,0.777778


In [25]:
log_model_classRep = classification_report(y_test,y_log_pred_test)
grd_model_classRep = classification_report(y_test,y_grd_pred_test)

print(f"Logistic Regression Classification Report:\n{log_model_classRep}")
print(f"Hypertuned Logistic Regression Classification Report:\n{grd_model_classRep}")

Logistic Regression Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.85      0.83        80
           1       0.80      0.76      0.78        63

    accuracy                           0.81       143
   macro avg       0.81      0.81      0.81       143
weighted avg       0.81      0.81      0.81       143

Hypertuned Logistic Regression Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.82      0.82        80
           1       0.78      0.78      0.78        63

    accuracy                           0.80       143
   macro avg       0.80      0.80      0.80       143
weighted avg       0.80      0.80      0.80       143



In [26]:
confusMat_test = confusion_matrix(y_log_pred_test,y_test)
fig = px.imshow(confusMat_test,text_auto=True,color_continuous_scale='RdBu_r',title="Confusion Matrix Heatmap Logistic Regression (Test)")
fig.show()

In [27]:
confusMat_test = confusion_matrix(y_grd_pred_test,y_test)
fig = px.imshow(confusMat_test,text_auto=True,color_continuous_scale='RdBu_r',title="Confusion Matrix Heatmap Hypertuned Logistic Regression (Test)")
fig.show()

In [28]:
y_test_log_scores = log_model.predict_proba(X_test)[:, 1]
y_test_grd_scores = gridOpti_model.predict_proba(X_test)[:, 1]

fpr_log, tpr_log, thresholds_log = roc_curve(y_test, y_test_log_scores)
auc_train_log_score = auc(fpr_log, tpr_log)
fpr_grd, tpr_grd, thresholds_grd = roc_curve(y_test, y_test_grd_scores)
auc_train_grd_score = auc(fpr_grd, tpr_grd)

In [29]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=fpr_log, y=tpr_log, mode='lines', name='ROC Curve'))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Blind Guess', line=dict(dash='dash')))

fig.update_layout(
    title=f'Logistic Regression ROC Curve for Test Data (AUC = {round(auc_train_log_score, 3)})',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    width=600,
    height=400
)

In [30]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=fpr_grd, y=tpr_grd, mode='lines', name='ROC Curve'))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Blind Guess', line=dict(dash='dash')))

fig.update_layout(
    title=f'Hypertuned Logistic Regression ROC Curve for Test Data (AUC = {round(auc_train_grd_score, 3)})',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    width=600,
    height=400
)

In [31]:
titanic_eval = {
    "features":X_test.columns.tolist(),
    "log_model":log_model.coef_[0].tolist(),
    "grid_model":gridOpti_model.coef_[0].tolist()
}
titanic_eval_df = pd.DataFrame(titanic_eval)
titanic_eval_df

,features,log_model,grid_model
0,sex,-2.482330,-1.742090
1,class_First,1.358583,0.779164
2,class_Second,-0.100878,0.027805
3,class_Third,-1.258289,-0.805393
4,embark_town_Cherbourg,0.261105,0.148058
5,embark_town_Queenstown,-0.333740,-0.093712
6,embark_town_Southampton,0.072051,-0.052771
7,age,-0.502898,-0.350710
8,sibsp,-0.055320,-0.073246
9,parch,-0.025464,0.003475


In [32]:
titanic_eval_df_long = titanic_eval_df.melt(id_vars="features",var_name="models", value_name="coefficients")
fig_scat = px.scatter(
        titanic_eval_df_long,
        x="features",
        y="coefficients",
        title=f"Coefficients of each feature per model",
        color="models")
fig_scat.update_layout(xaxis_title="Features", yaxis_title="Coefficients")
fig_scat